# FCCL 三產線機械式厚度量測 — CSV 合成資料產生器

**日期範圍：自動以上個月為結束月份，往前涵蓋完整 12 個月。** 例如執行當月為 2026-10，資料期間為 2025-10 至 2026-09。

**範圍：只產生資料，不訓練 AI 模型。** 依據上傳的四份 PDF 規格與產能規劃，涵蓋 1L / 2L / 3L 共 10 個產品型號。**每個批次只量測 10 個橫向位置，CSV 每列代表一個生產批次。**

- 月目標面積：1L 100,000、2L 250,000、3L 150,000 m²；產品比例依產線排程報告。
- 製程參數目標值與總厚度公差依《FCCL 各產品型號設備技術參數設定要求》；產量是**排程目標**而非真實實績。
- 厚度量測法設定為**機械式接觸量測的模擬值**；附件中的 WS-07 雷射測厚規格僅用來參考產品總厚度公差，**不把雷射測厚當作機械式量測紀錄**。
- 異常型態、參數擾動與根因為**合成標籤與教學假設**，不可視為經工廠驗證的因果關係。
- 設定 `MONTHS`、`SEED`、`ANOMALY_RATE` 可重複產生資料。預設 12 個月，輸出 UTF-8 BOM CSV。

In [ ]:
import numpy as np
import pandas as pd
from calendar import monthrange
from math import ceil
from pathlib import Path
import json
SEED = 42
# 固定資料期間，讓相同種子可重現同一批次與月份。
START_MONTH = "2025-10"
MONTHS = 12
ANOMALY_RATE = 0.18
OUTPUT_CSV = "fccl_thickness_dataset.csv"
rng = np.random.default_rng(SEED)
rng_ws04 = np.random.default_rng(SEED + 804)

In [ ]:
PRODUCT_SPECS = {'FCCL-3L-35/20/35': {'line': '3L', 'nominal': 160.0, 'tol': 2.5, 'speed': 3.8, 'unwind': 65.0, 'viscosity': 4800.0, 'gap': 135.0, 'pump': 96.0, 'wet': 56.0, 'temp': None, 'pressure': 340.0, 'cure': 165.0, 'rewind': 70.0}, 'FCCL-3L-18/12/18': {'line': '3L', 'nominal': 72.5, 'tol': 1.5, 'speed': 5.5, 'unwind': 38.0, 'viscosity': 4800.0, 'gap': 95.0, 'pump': 83.0, 'wet': 33.6, 'temp': None, 'pressure': 260.0, 'cure': 165.0, 'rewind': 43.0}, 'FCCL-2L-18/50/18': {'line': '2L', 'nominal': 86.0, 'tol': 1.5, 'speed': 5.5, 'unwind': 38.0, 'viscosity': 6200.0, 'gap': 160.0, 'pump': 520.0, 'wet': 210.0, 'temp': 365.0, 'pressure': 260.0, 'cure': 210.0, 'rewind': 43.0}, 'FCCL-2L-18/25/18': {'line': '2L', 'nominal': 61.0, 'tol': 1.5, 'speed': 5.5, 'unwind': 38.0, 'viscosity': 8500.0, 'gap': 85.0, 'pump': 260.0, 'wet': 105.0, 'temp': 395.0, 'pressure': 260.0, 'cure': 260.0, 'rewind': 43.0}, 'FCCL-2L-12/25/12': {'line': '2L', 'nominal': 49.0, 'tol': 1.0, 'speed': 7.2, 'unwind': 22.0, 'viscosity': 6200.0, 'gap': 85.0, 'pump': 340.0, 'wet': 105.0, 'temp': 365.0, 'pressure': 190.0, 'cure': 210.0, 'rewind': 27.0}, 'FCCL-2L-12/20/12': {'line': '2L', 'nominal': 44.0, 'tol': 1.0, 'speed': 7.2, 'unwind': 22.0, 'viscosity': 6200.0, 'gap': 85.0, 'pump': 272.0, 'wet': 84.0, 'temp': 365.0, 'pressure': 190.0, 'cure': 210.0, 'rewind': 27.0}, 'FCCL-2L-12/12/12': {'line': '2L', 'nominal': 36.0, 'tol': 1.0, 'speed': 7.2, 'unwind': 22.0, 'viscosity': 6200.0, 'gap': 85.0, 'pump': 163.0, 'wet': 50.4, 'temp': 385.0, 'pressure': 190.0, 'cure': 210.0, 'rewind': 27.0}, 'FCCL-2L-09/12/09': {'line': '2L', 'nominal': 30.0, 'tol': 1.0, 'speed': 7.2, 'unwind': 22.0, 'viscosity': 6200.0, 'gap': 85.0, 'pump': 163.0, 'wet': 50.4, 'temp': 365.0, 'pressure': 190.0, 'cure': 210.0, 'rewind': 27.0}, 'FCCL-1L-18/25': {'line': '1L', 'nominal': 55.0, 'tol': 1.5, 'speed': 5.5, 'unwind': 38.0, 'viscosity': 6200.0, 'gap': 85.0, 'pump': 260.0, 'wet': 105.0, 'temp': 365.0, 'pressure': 260.0, 'cure': 210.0, 'rewind': 43.0}, 'FCCL-1L-12/12': {'line': '1L', 'nominal': 24.0, 'tol': 1.0, 'speed': 7.2, 'unwind': 22.0, 'viscosity': 6200.0, 'gap': 85.0, 'pump': 163.0, 'wet': 50.4, 'temp': 365.0, 'pressure': 190.0, 'cure': 210.0, 'rewind': 27.0}}

# 月目標面積及排程產品組合；2L 按報告逐型號面積配置。
MONTHLY_AREA = {
"1L":{"FCCL-1L-18/25":40000,"FCCL-1L-12/12":60000},
"2L":{"FCCL-2L-12/20/12":70000,"FCCL-2L-12/25/12":70000,"FCCL-2L-09/12/09":20000,"FCCL-2L-12/12/12":45000,"FCCL-2L-18/25/18":25000,"FCCL-2L-18/50/18":20000},
"3L":{"FCCL-3L-18/12/18":112500,"FCCL-3L-35/20/35":37500}}
# 依報告建議經濟批量的近似面積（可自行修改）
BATCH_AREA = {"1L":10000,"2L":21000,"3L":10200}
LINE_WIDTH_M = {"1L":0.5,"2L":1.05,"3L":0.85}
assert len(PRODUCT_SPECS)==10
assert {k:sum(v.values()) for k,v in MONTHLY_AREA.items()}=={"1L":100000,"2L":250000,"3L":150000}
# Load WS-04 zone targets and control limits from the product equipment specification.
equipment_specs_path = next(
    (p / "data" / "processed" / "fccl_equipment_specs.json"
     for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "processed" / "fccl_equipment_specs.json").exists()),
    None,
)
if equipment_specs_path is None:
    raise FileNotFoundError("找不到 data/processed/fccl_equipment_specs.json，無法產生 WS-04 溫區資料。")
with equipment_specs_path.open("r", encoding="utf-8") as stream:
    equipment_specs = json.load(stream)
WS04_ZONE_PARAMETER_NAMES = {
    "dry_zone_1_2": "Zone 1-2 初級低溫揮發區",
    "dry_zone_3_4": "Zone 3-4 中級濃縮區溫度",
    "dry_zone_5_6": "Zone 5-6 預固化/凝膠區",
}
WS04_ZONE_SPECS = {}
for product_model, product_spec in equipment_specs.items():
    ws04 = next(station for station in product_spec["stations"] if station["code"] == "WS-04")
    WS04_ZONE_SPECS[product_model] = {
        key: next(parameter for parameter in ws04["parameters"] if name in parameter["name"])
        for key, name in WS04_ZONE_PARAMETER_NAMES.items()
    }

# WS-05 temperature means imidization temperature for 1L/2L products and
# hot-roller surface temperature for 3L products; both are defined in the spec.
WS05_TEMPERATURE_SPECS = {}
for product_model, product_spec in equipment_specs.items():
    ws05 = next(station for station in product_spec["stations"] if station["code"] == "WS-05")
    WS05_TEMPERATURE_SPECS[product_model] = next(
        parameter for parameter in ws05["parameters"]
        if "亞胺化核心峰值溫度" in parameter["name"]
        or "壓合鏡面鋼輥表面溫度" in parameter["name"]
    )

# WS-09 is the final inspection laboratory; populate all recipe measurements from its per-product specification.
WS09_PARAMETER_KEYS = (
    "ws09_peel_strength",
    "ws09_solder_temp",
    "ws09_dielectric_constant",
)
WS09_PARAMETER_SPECS = {}
for product_model, product_spec in equipment_specs.items():
    ws09 = next(station for station in product_spec["stations"] if station["code"] == "WS-09")
    if len(ws09["parameters"]) != len(WS09_PARAMETER_KEYS):
        raise ValueError(f"{product_model} 的 WS-09 規格數量不符。")
    WS09_PARAMETER_SPECS[product_model] = dict(zip(WS09_PARAMETER_KEYS, ws09["parameters"]))
rng_ws09 = np.random.default_rng(SEED + 909)


In [ ]:
def build_batches():
    rows=[]
    for month in pd.period_range(START_MONTH, periods=MONTHS, freq="M"):
        for line, models in MONTHLY_AREA.items():
            for model, target in models.items():
                n=ceil(target/BATCH_AREA[line])
                # 保留每型號月產能總量；最後一批允許小於經濟批量
                areas=[BATCH_AREA[line]]*(n-1)+[target-BATCH_AREA[line]*(n-1)]
                for i,area in enumerate(areas,1):
                    rows.append(dict(Month=str(month),Line=line,Product_Model=model,
                        Batch_ID=f"{month.strftime('%Y%m')}-{line}-{model.split('-')[-1].replace('/','')}-{i:03d}",
                        Batch_Area_m2=area, Batch_Length_m=round(area/LINE_WIDTH_M[line],2),
                        Model_Batch_Sequence=i))
    df=pd.DataFrame(rows)
    df["Sample_Date"] = ""
    # 日期是月內的模擬取樣日期，非實際設備排程時間；依報告組別大致先後排序
    order={"1L":["18/25","12/12"],"2L":["12/20/12","12/25/12","09/12/09","12/12/12","18/25/18","18/50/18"],"3L":["18/12/18","35/20/35"]}
    for (month,line),idx in df.groupby(["Month","Line"]).groups.items():
        ids=sorted(idx,key=lambda j:(order[line].index(df.at[j,"Product_Model"].split('-',2)[2]),df.at[j,"Model_Batch_Sequence"]))
        maxday=27 if line!="2L" else 28
        for rank,j in enumerate(ids):
            day=1+int(rank*maxday/len(ids))
            df.at[j,"Sample_Date"]=f"{month}-{day:02d}"
    return df
batches=build_batches()
print('批次數:',len(batches))
print(batches.groupby(['Month','Line'])['Batch_Area_m2'].sum().head(9))

In [ ]:
def make_record(batch):
    sp=PRODUCT_SPECS[batch.Product_Model]
    nominal,tol=sp['nominal'],sp['tol']
    # 依各產品 WS-05 規格設定實際溫度；1L/2L 為亞胺化，3L 為熱滾輪表面溫度。
    process={}
    for k in ['speed','unwind','viscosity','gap','pump','wet','temp','pressure','cure','rewind']:
        v=sp[k]
        if k == 'temp':
            v=float(WS05_TEMPERATURE_SPECS[batch.Product_Model]['target'])
        process[k+'_target']=v
        process[k+'_actual']=round(float(v+np.clip(rng.normal(0,max(abs(v)*0.003,0.03)),-abs(v)*0.006,abs(v)*0.006)),3) if v is not None else np.nan
    # Generate separate WS-04 zone readings from their product-specific QCP targets.
    for key, zone_spec in WS04_ZONE_SPECS[batch.Product_Model].items():
        target = float(zone_spec["target"])
        noise = np.clip(rng_ws04.normal(0, target * 0.003), -target * 0.006, target * 0.006)
        process[f"{key}_target"] = target
        process[f"{key}_actual"] = round(target + float(noise), 3)
    # Generate WS-09 lab result fields from the model-specific final-inspection recipe; values are synthetic demonstrations.
    for key, parameter_spec in WS09_PARAMETER_SPECS[batch.Product_Model].items():
        target = float(parameter_spec["target"])
        lower, upper = float(parameter_spec["lower"]), float(parameter_spec["upper"])
        noise_limit = min(target - lower, upper - target)
        noise = np.clip(rng_ws09.normal(0, noise_limit / 3), -noise_limit * 0.6, noise_limit * 0.6)
        process[f"{key}_target"] = target
        process[f"{key}_actual"] = round(target + float(noise), 4)
    anomaly=rng.random()<ANOMALY_RATE
    pattern=str(rng.choice(['Wedge','Center_Bulge','Mean_Shift','Local_Spike','Drift'],p=[.26,.23,.24,.17,.10])) if anomaly else 'Normal'
    x=np.linspace(-1,1,10)
    # 同一批次的十點共享批次偏移，另外有點位測量雜訊
    profile=nominal+rng.normal(0,tol*.12)+rng.normal(0,tol*.14,10)
    cause='No_Injected_Anomaly'
    if pattern=='Wedge':
        profile+=rng.choice([-1,1])*rng.uniform(1.0,1.7)*tol*x
        cause='Die_Gap_Imbalance_Hypothesis'
        process['gap_actual']=round(process['gap_actual']+rng.choice([-1,1])*2,3)
    elif pattern=='Center_Bulge':
        profile+=rng.choice([-1,1])*rng.uniform(1.1,1.9)*tol*(1-x*x)
        cause='Pressure_or_Drying_Nonuniformity_Hypothesis'
        if not np.isnan(process['pressure_actual']):process['pressure_actual']+=8
        process['dry_zone_3_4_actual']=round(process['dry_zone_3_4_target']*rng_ws04.choice([.97,1.03]),3)
    elif pattern=='Mean_Shift':
        profile+=rng.choice([-1,1])*rng.uniform(1.2,2.1)*tol
        cause='Pump_Flow_or_Viscosity_Hypothesis'
        process['pump_actual']=round(process['pump_actual']*rng.choice([.97,1.03]),3)
    elif pattern=='Local_Spike':
        profile[int(rng.integers(0,10))]+=rng.choice([-1,1])*rng.uniform(1.5,2.4)*tol
        # 局部突點以模頭開縫偏差作為可觀察的設備訊號。
        process['gap_actual']=round(process['gap_target']*rng.choice([.97,1.03]),3)
        cause='Die_Lip_Gap_or_Local_Contamination_Hypothesis'
    elif pattern=='Drift':
        profile+=rng.choice([-1,1])*np.linspace(0,2.1*tol,10)
        # 漸進漂移以收捲張力偏差代表；確保設備特徵可觀察到異常。
        process['rewind_actual']=round(process['rewind_target']*rng.choice([.97,1.03]),3)
        process['dry_zone_5_6_actual']=round(process['dry_zone_5_6_target']*rng_ws04.choice([.97,1.03]),3)
        cause='Gradual_Tension_or_Temperature_Drift_Hypothesis'
    profile=np.round(profile,2)
    record=batch._asdict()
    record.update({'Measurement_Method':'Mechanical_Contact_Simulated','Nominal_Thickness_um':nominal,
        'Tolerance_um':tol,'LSL_um':nominal-tol,'USL_um':nominal+tol,
        'Injected_Anomaly':int(anomaly),'Anomaly_Pattern':pattern,'Potential_Root_Cause_Synthetic':cause})
    record.update({f'Point_{i}_um':float(v) for i,v in enumerate(profile,1)})
    record.update({'Thickness_Mean_um':round(float(profile.mean()),3),'Thickness_Std_um':round(float(profile.std(ddof=1)),3),
        'Thickness_Range_um':round(float(np.ptp(profile)),3),
        'QC_Result':'NG' if np.any((profile<nominal-tol)|(profile>nominal+tol)) else 'PASS'})
    record.update(process)
    return record

df=pd.DataFrame([make_record(b) for b in batches.itertuples(index=False)])
assert df.Batch_ID.is_unique
assert df[[f'Point_{i}_um' for i in range(1,11)]].notna().all().all()
assert df[[f"{key}_{kind}" for key in WS09_PARAMETER_KEYS for kind in ("target", "actual")]].notna().all().all()
assert df.groupby(['Month','Line']).Batch_Area_m2.sum().unstack().eq(pd.Series({'1L':100000,'2L':250000,'3L':150000})).all().all()
df.to_csv(OUTPUT_CSV,index=False,encoding='utf-8-sig')
print('已輸出:',OUTPUT_CSV,'資料列數:',len(df),'欄位數:',len(df.columns))
print(df.groupby('Line').size())
print(df[['Batch_ID','Sample_Date','Product_Model','Thickness_Mean_um','QC_Result','Anomaly_Pattern']].head(10).to_string(index=False))

## 解讀與限制

1. 每列為一批、10 點為同一批橫向位置；不可將 10 點誤解為 10 批。
2. `Injected_Anomaly` 是模擬注入標籤，`QC_Result` 是依厚度規格判斷，兩者可能不同（例如有形狀異常但十點尚未超規）。
3. `Potential_Root_Cause_Synthetic` 是設計假說，不代表現場確診。
4. 各型號每月面積符合附件目標，但小尾批、取樣日期、參數波動和異常率均為模擬假設，沒有宣稱逐時產線排程可行。
5. 附件中的某些總厚度與層結構名義加總不一致，**本程式沿用附件明列的總厚度，不自行改算**。